In [ ]:
#imports
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
import random
layers = tf.keras.layers
Model = tf.keras.Model

In [ ]:
# Extract the CUB-200 dataset (only need to run once)
import os
if not os.path.exists("data/CUB_200_2011/images"):
    !mkdir -p data
    !tar -xzf CUB_200_2011.tgz -C data
    print("Dataset extracted.")
else:
    print("Dataset already extracted.")

In [ ]:

print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

In [ ]:
print(tf.config.experimental.list_physical_devices('GPU'))
import subprocess
print(subprocess.run(['nvidia-smi', 'topo', '-m'], capture_output=True, text=True).stdout)

In [ ]:
def load_and_resize(path1, path2, label):
    img1 = tf.io.read_file(path1)
    img2 = tf.io.read_file(path2)
    img1 = tf.image.decode_png(img1, channels=3)
    img2 = tf.image.decode_png(img2, channels=3)
    img1 = tf.image.resize(img1, (224, 224))
    img2 = tf.image.resize(img2, (224, 224))
    img1 = tf.cast(img1, tf.float32) / 255.0
    img2 = tf.cast(img2, tf.float32) / 255.0
    return (img1, img2), label

def augment_img(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.image.random_saturation(img, lower=0.85, upper=1.15)
    # keep values valid after brightness/contrast/saturation jitter
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img

def augment_pair(imgs, label):
    img1, img2 = imgs
    img1 = augment_img(img1)
    img2 = augment_img(img2)
    return (img1, img2), label

def create_dataset(pairs, batch_size=8, training=False):
    img1_paths = [p[0] for p in pairs]
    img2_paths = [p[1] for p in pairs]
    labels     = [p[2] for p in pairs]
    dataset = tf.data.Dataset.from_tensor_slices(
        ((img1_paths, img2_paths), labels)
    )
    options = tf.data.Options()
    options.experimental_deterministic = False
    dataset = dataset.with_options(options)

    # 1. Decode + resize (deterministic, safe to cache)
    dataset = dataset.map(
        lambda paths, label: load_and_resize(paths[0], paths[1], label),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    # 3. Shuffle
    dataset = dataset.shuffle(buffer_size=1000, reshuffle_each_iteration=True)

    # 4. Augment AFTER caching, so it's randomized fresh every epoch (training only)
    if training:
        dataset = dataset.map(augment_pair, num_parallel_calls=tf.data.AUTOTUNE)

    # 5. Batch
    dataset = dataset.batch(batch_size, drop_remainder=True)

    # 6. Prefetch for the GPU
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
import random

def make_pairs(char_dict, num_pairs=20):
    class_names = list(char_dict.keys())
    all_pairs = []
    for current_class in class_names:
        images = char_dict[current_class]
        if len(images) < 2:
            continue  # skip species with too few images to form a same-pair

        # same pairs (label = 1)
        used_same = set()
        attempts = 0
        max_attempts = num_pairs * 20  # safety valve for small image counts
        while len(used_same) < num_pairs and attempts < max_attempts:
            img1 = random.choice(images)
            img2 = random.choice(images)
            attempts += 1
            if img2 == img1:
                continue
            pair_check = tuple(sorted([img1, img2]))
            if pair_check in used_same:
                continue
            used_same.add(pair_check)
            all_pairs.append([img1, img2, 1])

        # different pairs (label = 0), drawing class2 from a wide pool each time
        used_diff = set()
        attempts = 0
        while len(used_diff) < num_pairs and attempts < max_attempts:
            class2 = random.choice([x for x in class_names if x != current_class])
            img1 = random.choice(images)
            img2 = random.choice(char_dict[class2])
            attempts += 1
            pair_check = tuple(sorted([img1, img2]))
            if pair_check in used_diff:
                continue
            used_diff.add(pair_check)
            all_pairs.append([img1, img2, 0])

    random.shuffle(all_pairs)
    return all_pairs

In [ ]:
layers = tf.keras.layers
Model = tf.keras.Model
regularizers = tf.keras.regularizers

def build_encoder(input_shape=(224, 224, 3), embedding_dim=128, freeze_backbone=False, l2_reg=1e-4):
    inputs = layers.Input(shape=input_shape, name="image")
    base_model = tf.keras.applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
        pooling='avg'
    )
    base_model.trainable = not freeze_backbone
    x = base_model(inputs)
    x = layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(embedding_dim, kernel_regularizer=regularizers.l2(l2_reg))(x)
    embeddings = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1))(x)
    return Model(inputs, embeddings)

def build_siamese_model(enc):
    img1 = layers.Input(shape=(224, 224, 3), name='img1')
    img2 = layers.Input(shape=(224, 224, 3), name='img2')
    tensor1 = enc(img1)
    tensor2 = enc(img2)
    distance = layers.Lambda(lambda tensors: tf.norm(tensors[0] - tensors[1], axis=1))([tensor1, tensor2])
    return Model([img1, img2], distance)

In [ ]:
import os
import random
from sklearn.model_selection import train_test_split

def load_and_split_cub_images(images_dir, test_size=0.2, val_size=0.1):
    """
    Loads CUB-200 images and splits species into train/val/test sets.
    val_size is the fraction of the *original* total reserved for validation
    (taken out of the remaining train portion after the test split).
    """
    all_species = [d for d in os.listdir(images_dir) if os.path.isdir(os.path.join(images_dir, d))]

    # First split off the test species
    train_val_species, test_species = train_test_split(
        all_species, test_size=test_size, random_state=42
    )

    # Then split the remainder into train/val
    # Adjust val_size so it represents the right fraction of the original whole
    relative_val_size = val_size / (1 - test_size)
    train_species, val_species = train_test_split(
        train_val_species, test_size=relative_val_size, random_state=42
    )

    def get_images_for_species(species_list):
        data = {}
        for species in species_list:
            species_path = os.path.join(images_dir, species)
            full_paths = [
                os.path.join(species_path, f)
                for f in os.listdir(species_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))
            ]
            if full_paths:
                data[species] = full_paths
        return data

    return (
        get_images_for_species(train_species),
        get_images_for_species(val_species),
        get_images_for_species(test_species),
    )

# 1. Load and Split
cub_path = 'data/CUB_200_2011/images'
train_dict, val_dict, test_dict = load_and_split_cub_images(cub_path, test_size=0.2, val_size=0.1)
print(f"Species for training: {len(train_dict)}, validation: {len(val_dict)}, testing: {len(test_dict)}")

# 2. Generate Pairs
train_pairs = make_pairs(train_dict, num_pairs=40)
val_pairs = make_pairs(val_dict, num_pairs=10)
test_pairs = make_pairs(test_dict, num_pairs=10)  # Fewer pairs for testing

# 3. Create Datasets
train_dataset = create_dataset(train_pairs, batch_size=16, training=True)
val_dataset = create_dataset(val_pairs, batch_size=16, training=False)
test_dataset = create_dataset(test_pairs, batch_size=16, training=False)

print(f"Ready: {len(train_pairs)} training pairs, {len(val_pairs)} validation pairs, {len(test_pairs)} test pairs.")

In [ ]:
def build_encoder(input_shape=(224, 224, 3), embedding_dim=128, freeze_backbone=False, l2_reg=1e-4):
    ...
    return Model(inputs, embeddings, name='encoder')

def build_siamese_model(enc):
    ...
    distance = EuclideanDistance()([tensor1, tensor2])
    return Model([img1, img2], distance)

In [ ]:
enc     = build_encoder()
siamese = build_siamese_model(enc)
siamese.load_weights('best_model.weights.h5')
enc = siamese.get_layer('encoder')

initial_lr = 1e-4
optimizer  = tf.keras.optimizers.Adam(initial_lr)
enc.summary()
siamese.summary()

In [ ]:
def compute_embeddings(encoder, char_dict, sample_per_class=8):
    cache = {}
    for class_name, images in char_dict.items():
        sampled = random.sample(images, min(sample_per_class, len(images)))
        for img_path in sampled:
            img = tf.io.read_file(img_path)
            img = tf.image.decode_image(img, channels=3, expand_animations=False)
            img = tf.image.resize(img, (224, 224))
            img = tf.cast(img, tf.float32) / 255.0
            img = tf.expand_dims(img, 0)
            emb = encoder(img, training=False).numpy()[0]
            cache[img_path] = emb
    return cache


def make_pairs_with_hard_negatives(char_dict, embeddings_cache=None,
                                   num_pairs=40, hard_neg_ratio=0.5):
    class_names  = list(char_dict.keys())
    all_pairs    = []
    num_hard     = int(num_pairs * hard_neg_ratio) if embeddings_cache else 0
    num_random   = num_pairs - num_hard
    max_attempts = num_pairs * 20

    for current_class in class_names:
        images = char_dict[current_class]
        if len(images) < 2:
            continue

        # same pairs (label = 1)
        used_same = set()
        attempts  = 0
        while len(used_same) < num_pairs and attempts < max_attempts:
            img1, img2 = random.choice(images), random.choice(images)
            attempts  += 1
            if img1 == img2:
                continue
            key = tuple(sorted([img1, img2]))
            if key in used_same:
                continue
            used_same.add(key)
            all_pairs.append([img1, img2, 1])

        used_diff = set()

        # random negatives (label = 0)
        attempts = 0
        while len(used_diff) < num_random and attempts < max_attempts:
            class2 = random.choice([x for x in class_names if x != current_class])
            img1   = random.choice(images)
            img2   = random.choice(char_dict[class2])
            attempts += 1
            key = tuple(sorted([img1, img2]))
            if key in used_diff:
                continue
            used_diff.add(key)
            all_pairs.append([img1, img2, 0])

        # hard negatives (label = 0)
        if num_hard > 0 and embeddings_cache:
            anchors = [(p, embeddings_cache[p]) for p in images if p in embeddings_cache]
            if not anchors:
                continue

            other_classes   = [x for x in class_names if x != current_class]
            sampled_classes = random.sample(other_classes, min(30, len(other_classes)))
            candidates      = [
                (p, embeddings_cache[p])
                for cls in sampled_classes
                for p   in char_dict[cls]
                if p in embeddings_cache
            ]
            if not candidates:
                continue

            cand_paths = [c[0] for c in candidates]
            cand_embs  = np.stack([c[1] for c in candidates])

            hard_found = 0
            for anchor_path, anchor_emb in anchors:
                if hard_found >= num_hard:
                    break
                dists = np.linalg.norm(cand_embs - anchor_emb, axis=1)
                for idx in np.argsort(dists):
                    key = tuple(sorted([anchor_path, cand_paths[idx]]))
                    if key in used_diff:
                        continue
                    used_diff.add(key)
                    all_pairs.append([anchor_path, cand_paths[idx], 0])
                    hard_found += 1
                    break

    random.shuffle(all_pairs)
    return all_pairs

In [ ]:
import numpy as np
from tqdm.auto import tqdm

FREEZE_EPOCHS   = 5     # epochs 0-4: head only, backbone fully frozen
TOTAL_EPOCHS    = 20
WARMUP_EPOCHS   = 3     # hard-neg mining kicks in from here, unchanged

# Staged unfreeze: 5 layers added at each of these epoch boundaries
UNFREEZE_STAGE_EPOCHS = [5, 8, 11, 14]     # 4 stages, 3 epochs apart
LAYERS_PER_STAGE      = 5
STAGE_LRS             = [1e-5, 5e-6, 2e-6, 1e-6]   # progressively gentler as more params unlock

def contrastive_loss(labels, distances, margin=1.0):
    labels = tf.cast(labels, tf.float32)
    same   = labels       * tf.square(distances)
    diff   = (1 - labels) * tf.square(tf.maximum(margin - distances, 0))
    return same + diff

def _train_step(batch):
    (img1, img2), labels = batch
    with tf.GradientTape() as tape:
        distances = siamese([img1, img2], training=True)
        loss = tf.reduce_mean(contrastive_loss(labels, distances))
    grads = tape.gradient(loss, siamese.trainable_variables)
    optimizer.apply_gradients(zip(grads, siamese.trainable_variables))
    return loss

def _val_step(batch):
    (img1, img2), labels = batch
    distances = siamese([img1, img2], training=False)
    return tf.reduce_mean(contrastive_loss(labels, distances))

train_step = tf.function(_train_step)
val_step   = tf.function(_val_step)

def unfreeze_next_stage(base_model, n_layers, stage_idx):
    """
    Unfreezes the next `n_layers` non-BatchNorm layers, counting from the
    deepest (last) layers inward. BatchNorm stays frozen throughout --
    fine-tuning BN running stats on small batches (16 here) is a common
    source of instability and isn't worth the risk for this dataset size.
    """
    trainable_layers = [
        l for l in base_model.layers
        if not isinstance(l, tf.keras.layers.BatchNormalization)
    ]
    total_to_unfreeze = n_layers * (stage_idx + 1)
    target_layers = trainable_layers[-total_to_unfreeze:]
    for layer in target_layers:
        layer.trainable = True
    print(f"  Stage {stage_idx + 1}: unfroze last {total_to_unfreeze} non-BN layers "
          f"({[l.name for l in target_layers[-n_layers:]]})")

patience_counter    = 0
train_loss_history  = []
val_loss_history    = []
best_val_loss       = float("inf")
embeddings_cache    = None
current_stage       = -1   # -1 = fully frozen, 0..3 = which stage we're in

print("Starting Training...\n")
print(f"Trainable vars: {len(siamese.trainable_variables)}")
for epoch in range(TOTAL_EPOCHS):
    print(f"Epoch {epoch + 1}/{TOTAL_EPOCHS}")

    # -- Freeze / staged unfreeze ---
    if epoch == 0:
        enc.get_layer('resnet50').trainable = True   # allow layer-level control below
        for layer in enc.get_layer('resnet50').layers:
            layer.trainable = False                   # start fully frozen
        print("  Backbone: FROZEN (head only)")
    elif epoch in UNFREEZE_STAGE_EPOCHS:
        current_stage += 1
        unfreeze_next_stage(enc.get_layer('resnet50'), LAYERS_PER_STAGE, current_stage)
        new_lr = STAGE_LRS[current_stage]
        optimizer = tf.keras.optimizers.Adam(new_lr)
        train_step = tf.function(_train_step)   # retrace: new trainable set + new optimizer
        val_step   = tf.function(_val_step)
        print(f"  Backbone: STAGE {current_stage + 1} unfrozen  (LR -> {new_lr:.1e})")

    # -- Regenerate pairs every epoch ---
    hard_ratio  = 0.0 if epoch < WARMUP_EPOCHS else 0.5
    train_pairs = make_pairs_with_hard_negatives(
        train_dict, embeddings_cache=embeddings_cache,
        num_pairs=40, hard_neg_ratio=hard_ratio
    )
    val_pairs = make_pairs(val_dict, num_pairs=25)

    train_dataset = create_dataset(train_pairs, batch_size=16, training=True)
    val_dataset   = create_dataset(val_pairs,   batch_size=16, training=False)
    print(f"  Pairs -> {len(train_pairs)} train | {len(val_pairs)} val | hard_ratio={hard_ratio}")
    print(f"  Trainable vars: {len(siamese.trainable_variables)}")

    # -- Train ---
    batch_losses = []
    train_pbar   = tqdm(train_dataset, desc="  Training", unit="batch")
    for batch in train_pbar:
        loss     = train_step(batch)
        loss_val = loss.numpy()
        batch_losses.append(loss_val)
        train_pbar.set_postfix(loss=f"{loss_val:.4f}")
    avg_train_loss = float(np.mean(batch_losses))
    train_loss_history.append(avg_train_loss)

    # -- Validate ---
    val_losses = []
    val_pbar   = tqdm(val_dataset, desc="  Validation", unit="batch")
    for batch in val_pbar:
        val_losses.append(val_step(batch).numpy())
        val_pbar.set_postfix(loss=f"{val_losses[-1]:.4f}")
    avg_val_loss = float(np.mean(val_losses))
    val_loss_history.append(avg_val_loss)

    print(f"  Summary -> Train: {avg_train_loss:.6f} | Val: {avg_val_loss:.6f}")

    # -- Checkpoint + LR reduction ---
    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        patience_counter = 0
        siamese.save_weights("best_model.weights.h5")
        print("  -> Improvement! Saved best model.")
    else:
        patience_counter += 1
        print(f"  -> No improvement ({patience_counter}/3)")
        if patience_counter >= 3:
            new_lr = float(optimizer.learning_rate.numpy()) * 0.5
            optimizer.learning_rate.assign(new_lr)
            patience_counter = 0
            print(f"  -> Reducing LR to {new_lr:.2e}")

    # -- Update embedding cache ---
    if epoch >= WARMUP_EPOCHS - 1:
        print("  Computing embeddings for hard-neg mining...")
        embeddings_cache = compute_embeddings(enc, train_dict, sample_per_class=8)

print("\nTraining complete.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_loss_history) + 1), train_loss_history, label="Train Loss")
plt.plot(range(1, len(val_loss_history) + 1), val_loss_history, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss 2")
plt.legend()
plt.grid(True)
plt.savefig(
    "val_vs_train.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import json

# Extract just the species names (the keys) for the split record
split_record = {
    "train_species": list(train_dict.keys()),
    "val_species": list(val_dict.keys()),
    "test_species": list(test_dict.keys())
}

# Save to a JSON file in your working directory
with open("cub200_dataset_splits.json", "w") as f:
    json.dump(split_record, f, indent=4)
    
print("Splits successfully saved to JSON!")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_loss_history) + 1), train_loss_history, label="Train Loss")
plt.plot(range(1, len(val_loss_history) + 1), val_loss_history, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss 2")
plt.legend()
plt.grid(True)
plt.savefig(
    "val_vs_train.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()